In [1]:
import sys
import os
import json

import datasets
from transformers import AutoTokenizer

# Add the project root directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.agents.rapr_agent import RAPRAgent

### Agent setup

In [2]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")

agent = RAPRAgent(
    response_model_name="meta-llama/Meta-Llama-3.1-8B-Instruct",
    eval_model_name="gemini-2.0-flash", 
    temperature=0.5,
    tokenizer=tokenizer,
)

### SQUAD

In [3]:
dataset_name = "rajpurkar_squad"
path = f"../../../dataset/raw_model_responses/test/test_rajpurkar_squad.json"

data = datasets.load_dataset("json", data_files=path)
data = data["train"]

In [5]:
# final_results = []
# input_tokens = 0
# output_tokens = 0
# for sample in data:
#     question = sample["additional_info"]["question"]
#     context = sample["additional_info"]["context"]

#     corrected_responses = []

#     for i in range(len(sample["response"])):
#         response = sample["response"][i]
#         result = agent.model.invoke({
#             "question": question,
#             "context": context,
#             "model_response": response,
#         })
#         corrected_responses.append(result["corrected_response"] if "corrected_response" in result else response)
#         input_tokens += result["input_tokens"]
#         output_tokens += result["output_tokens"]
#         break

#     sample["response"] = corrected_responses
#     final_results.append(sample)
#     # break


final_results = []
input_tokens = 0
output_tokens = 0
for sample in data:
    question = sample["additional_info"]["question"]
    context = sample["additional_info"]["context"]
    corrected_responses = []

    for i in range(len(sample["response"])):
        result = agent.model.invoke({
            "question": question,
            "context": context,
        })
        corrected_responses.append(result["corrected_response"] if "corrected_response" in result else result["model_response"])
        input_tokens += result["input_tokens"]
        output_tokens += result["output_tokens"]

    sample["response"] = corrected_responses
    final_results.append(sample)

2025-10-22 10:24:37.862 | INFO     | src.agents.rapr_agent:generate_model_response:83 - Generated model response: St. John's Cathedral was constructed in the 14th century.
2025-10-22 10:24:38.427 | INFO     | src.agents.rapr_agent:generate_queries:110 - Generated queries: ["In what century was St. John's Cathedral constructed?", "When was St. John's Cathedral constructed?"]
2025-10-22 10:24:39.473 | INFO     | src.agents.rapr_agent:retrieve_evidence:142 - Retrieved evidence: ["St. John's Cathedral (14th century)", "St. John's Cathedral (14th century)"]
2025-10-22 10:24:39.476 | INFO     | src.agents.rapr_agent:revise:160 - Checking agreement for query: In what century was St. John's Cathedral constructed? and evidence: St. John's Cathedral (14th century)
2025-10-22 10:24:39.845 | INFO     | src.agents.rapr_agent:revise:178 - Should edit: False
2025-10-22 10:24:39.847 | INFO     | src.agents.rapr_agent:revise:160 - Checking agreement for query: When was St. John's Cathedral constructed?

In [6]:
print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {output_tokens}")
print(f"Average input tokens: {input_tokens / (len(final_results)*5)}")
print(f"Average output tokens: {output_tokens / (len(final_results)*5)}")

Input tokens: 352854
Output tokens: 7218
Average input tokens: 3714.2526315789473
Average output tokens: 75.97894736842105


In [5]:
1/0

ZeroDivisionError: division by zero

In [7]:
final_results

[{'task_info': {'dataset': 'rajpurkar_squad', 'type': 'Contextual QA'},
  'additional_info': {'answer': ['14th century',
    '14th century',
    '14th century'],
   'context': 'Gothic architecture is represented in the majestic churches but also at the burgher houses and fortifications. The most significant buildings are St. John\'s Cathedral (14th century), the temple is a typical example of the so-called Masovian gothic style, St. Mary\'s Church (1411), a town house of Burbach family (14th century), Gunpowder Tower (after 1379) and the Royal Castle Curia Maior (1407–1410). The most notable examples of Renaissance architecture in the city are the house of Baryczko merchant family (1562), building called "The Negro" (early 17th century) and Salwator tenement (1632). The most interesting examples of mannerist architecture are the Royal Castle (1596–1619) and the Jesuit Church (1609–1626) at Old Town. Among the first structures of the early baroque the most important are St. Hyacinth\'s 

In [8]:
with open(f"../../../dataset/rapr_responses/test_rapr_{dataset_name}.json", "w") as f:
    json.dump(final_results, f, indent=4)